In [6]:
# Import required libraries
import tensorflow as tf
from tensorflow.keras.datasets import reuters
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from keras.utils import plot_model

In [9]:
# Setting parameters for model and training
inputdim = 10000      # vocabulary size
inputlen = 200        # length of each padded sequence
batch_size = 32       # number of samples per gradient update during training
embedding_dim = 128   # embedding dimensions
epochs = 5            # number of training iterations over the dataset


In [8]:
# Loading data
print ('Loading data...')
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=inputdim)
print (len(x_train), 'train sequences')     # Number of training sequences
print (len(x_test), 'test sequences')       # Number of testing sequences

Loading data...
2110848/2110848 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
8982 train sequences
2246 test sequences


In [10]:
# Pad sequences to ensure uniform length for input to LSTM
x_train = pad_sequences(x_train, maxlen=inputlen)
x_test = pad_sequences(x_test, maxlen=inputlen)
print ('x_train shape:', x_train.shape)
print ('x_test shape:', x_test.shape)

x_train shape: (8982, 200)
x_test shape: (2246, 200)


In [12]:
# One-hot encoding the target labels for multi-class classification
y_encoded_train = to_categorical(y_train, num_classes=46)
y_encoded_test = to_categorical(y_test, num_classes=46)
print("{} becomes {}".format(y_train[3],y_encoded_train[3]))
print("{} becomes {}".format(y_test[40],y_encoded_test[40]))

4 becomes [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
15 becomes [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [20]:
from keras.src.layers.core.input_layer import Input
# Define the LSTM model
model = Sequential()      # Initialize a sequential model
model.add(Input(shape=(inputlen,)))
model.add(Embedding(inputdim, embedding_dim))    # Embedding layer: Turns positive integers into dense vectors of fixed size
model.add(LSTM(64))       # LSTM layer with 64 units
model.add(Dense(46, activation='softmax'))    # Output layer with softmax activation for multi-class classification

In [21]:
# Display the model architecture
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 46)             │         2,990 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,332,398 (5.08 MB)

 Trainable params: 1,332,398 (5.08 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model on training data
training_values = model.fit(x_train, y_encoded_train, batch_size=batch_size, epochs=epochs, validation_data=(x_test, y_encoded_test))

Epoch 1/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 39s 129ms/step - accuracy: 0.4606 - loss: 2.2061 - val_accuracy: 0.5307 - val_loss: 1.8234
Epoch 2/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 41s 131ms/step - accuracy: 0.5652 - loss: 1.7365 - val_accuracy: 0.5988 - val_loss: 1.6211
Epoch 3/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 35s 126ms/step - accuracy: 0.6188 - loss: 1.5261 - val_accuracy: 0.5525 - val_loss: 1.6180
Epoch 4/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 36s 128ms/step - accuracy: 0.6654 - loss: 1.3495 - val_accuracy: 0.6385 - val_loss: 1.4591
Epoch 5/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 37s 130ms/step - accuracy: 0.7132 - loss: 1.1318 - val_accuracy: 0.6683 - val_loss: 1.3563


In [23]:
# Evaluate model on test data
loss_value, accuracy = model.evaluate(x_test, y_encoded_test,
                            batch_size=batch_size)
print('Test score:', loss_value)
print('Test accuracy:', accuracy)

71/71 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.6683 - loss: 1.3563
Test score: 1.3563340902328491
Test accuracy: 0.6682991981506348
